<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Chicken_Counting/Chicken_Counting.ipynb)

# Chicken Counting

## **1. Problem Description**

As the leader of an AI research team collaborating with Silkie chicken farmers, you are tasked with solving a critical challenge in traditional free-range farming. Accurate counting of livestock is crucial for both farmers and insurance companies, as factors like disease outbreaks and predator invasions can significantly impact the survival rate of these chickens in a short time. While insurance coverage helps mitigate farming risks, the claims process requires precise counting of livestock losses. Your farmers have approached your team for help in developing more accurate, automated counting systems. The challenge before your research team is to develop an optimized Silkie chicken counting model using density estimation techniques that can provide reliable counts to support both farm management and insurance processes.

Your team has access to a pretrained feature extractor for Silkie chicken images, but you'll need to design and train the density estimation decoder to create a complete counting solution. Your task is to build upon this foundation by developing an effective decoder architecture and training strategy to achieve accurate chicken counts that farmers and insurance companies can rely on.

The below figure shows an image in the dataset, as well as the corresponding true density distribution and a predicted density distribution generated by the baseline model. The total density (sum of densities across all areas) is labeled.

<img src="./figs/Chicken Counting Fig 1.png" width="800">

## **2. Dataset**

The structure of the provided Silkie chicken image dataset is as follows:

```
datasets/
├── train/
│   └── A dataset with features:
│       ├── `image`: `PIL.Image` with RGB channels (3x720x1280)
│       └── `density`: a 2D array of shape 180x320
└── base.pth (Pretrained Model)


os.environ.get("DATA_PATH")/
├── test_a/
│   └── A dataset with features:
│       └── `image`: `PIL.Image` with RGB channels (3x720x1280)
└── test_b/
    └── A dataset with features:
        └── `image`: `PIL.Image` with RGB channels (3x720x1280)
```

(1) Training set location: `datasets`, files in this folder are used for model fine-tuning. It contains a train folder, which stores a dataset with 100 images and their corresponding density maps.

(2) Validation set (test_a) and Test set (test_b): These will be used to evaluate scores on Leaderboard A and Leaderboard B, respectively. They will be inaccessible to contestants. Only the score achieved on test set B will be used for final scoring.

(3) Datasets size:

- Training set: 100 images.
- Validation set: 100 images.
- Test set: 100 images.

(4) Validation set (test_a) and Test set (test_b) are not visible.

(5) Due to limits of computing resources, training density maps are reshaped to $1\times 180 \times 320$. **NOTE** the output density map can be viewed as a 2D real number matrix with shape $180\times 320$, the sum of all matrix values is the count of chickens.

## **3. Task**

Your task is to train your own model using the training data to predict density maps, thereby serving the purpose of chicken counting.

You may extend and optimize the given pretrained model to improve its count prediction accuracy. The pretrained model `base.pth` only contains the weights of the first four layers of the model (the feature extraction model), and the function `load_pretrained_weights_partial` in the baseline code can be used to load these partial weights into your model. You may construct a density decoder `DensityDecoder` and combine it with the pretrained feature extraction module to form a complete data prediction model.

```
class DensityDecoder(nn.Module):
    def __init__(self):
        #################################################
        # Your code here
        #################################################

    def forward(self, x):
        #################################################
        # Your code here
        #################################################
        return x
```

You can also build your own model without the pretrained model we provided.

This task is the continuation of Satellite Weather Forecasting. A kind remind is the UNET is easily to full GPU memory without any feature engineering. Then, the GPU memory error message will be reported.

Please follow these rules to achieve a score normally:

(1) Your model must output the predicted density map.

(2) Due to limits of computing resources, your output density map should be reshaped to $180 \times 320$. This is also the shape of target density maps provided in the train dataset.

## 4. Submission

Please submit a **submission.ipynb** that includes the following components:

（1）**Training Code**  

- Include the full training pipeline.

（2）**Evaluation Code**
- Evaluate your model on the validation set and test set.

- The output should be saved as **`submission.npz`**. This must be a valid `npz` file containing two arrays `pred_a` and `pred_b`, each with shape `100x1x180x320` (The evaluation script will also accept predictions in the shape of `100x180x320`, if you decide to squeeze the channel dimension).

  **Any result that does not meet the specified size will be considered invalid, resulting in an assessment score of zero.**
  
- Each element of the density map should be **no less than zero**, otherwise will result in an assessment score of zero.

## **5. Scoring**

You will be scored based on the mean relative error of your model. Relative error is defined by:

$$
\text{Relative Error} = \frac{|y_i - \hat{y}_i|}{|y_i|}
$$

where $y_i$ is the true total density for the $i$-th sample, and $\hat{y}_i$ is the predicted total density for the $i$-th sample.

Your final score before normalization will be calculated based on your mean relative error, as follows:

$$
\text{Score} = \exp(-\frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{y_i})
$$

## **6. Baseline & Training Set**

- Below you can find the baseline solution.
- The dataset is in `training_set` folder.
- The highest score by the Scientific Committee for this task is 0.89,  this score is used for score unification.
- The baseline score by the Scientific Committee for this task is 0.71, this score is used for score unification.

In [ ]:
import os
import cv2
import math
import json
import torch
import logging
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm.auto import tqdm
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
import torch.optim

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
TARGET_SIZE = (180, 320)
INPUT_SIZE = (720, 1280)

In [ ]:
class FeatureExtraction(nn.Module):
    def __init__(self):
        super(FeatureExtraction, self). __init__()
        resnet = models.resnet18(weights=None)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return x1, x2, x3, x4

class ChickenCounting(nn.Module):
    def __init__(self, base_weights_path=None):
        super(ChickenCounting, self).__init__()
        self.feature_extractor = FeatureExtraction()
        
        if base_weights_path and os.path.exists(base_weights_path):
            self.feature_extractor.load_state_dict(torch.load(base_weights_path, map_location='cpu'), strict=False)
            logging.info(f"Loaded base weights from {base_weights_path}")

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2) 
        self.conv1 = nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(128 + 128, 128, kernel_size=3, padding=1)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1)
        
        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        x1, x2, x3, x4 = self.feature_extractor(x)
        
        d1 = self.up1(x4)
        d1 = torch.cat([d1, x3], dim=1) 
        d1 = F.relu(self.conv1(d1))
        
        d2 = self.up2(d1)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = F.relu(self.conv2(d2))
        
        d3 = self.up3(d2)
        d3 = torch.cat([d3, x1], dim=1)
        d3 = F.relu(self.conv3(d3))
        
        out = self.final_conv(d3)
        
        if out.shape[2:] != TARGET_SIZE:
            out = F.interpolate(out, size=TARGET_SIZE, mode='bilinear', align_corners=False)
            
        return out

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss(reduction='sum')
    model.to(DEVICE)
    
    best_mae = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            inputs, targets = batch['image'].to(DEVICE), batch['density'].to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_mae = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs, targets = batch['image'].to(DEVICE), batch['density'].to(DEVICE)
                outputs = model(inputs)
                
                pred_count = outputs.sum().item()
                true_count = targets.sum().item()
                val_mae += abs(pred_count - true_count)
        
        avg_mae = val_mae / len(val_loader.dataset)
        logging.info(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val MAE: {avg_mae:.4f}")
        
        if avg_mae < best_mae:
            best_mae = avg_mae
            torch.save(model.state_dict(), "best_model.pth")
            
    return model

def predict_set(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            inputs = batch['image'].to(DEVICE)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
    return np.concatenate(all_preds, axis=0)

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Train2", 
                            data_dir="train",
                            split="train")  

image_transform = transforms.Compose([
    transforms.ToTensor(),
])

def collate_fn(batch, scale=100):
    return {
        "image": torch.stack([image_transform(item["image"]) for item in batch]),
        "density": torch.stack([torch.tensor(item["density"], dtype=DTYPE).unsqueeze(0) * scale for item in batch])
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = ChickenCounting(base_weights_path="/kaggle/working/model.pth")

In [ ]:
import torch.optim as optim

################################################################################
# Experiment Settings
################################################################################
learning_rate = 1e-4
lr_decay = 1e-5
weight_decay = 0.0001
save_path = "model.pth"

epochs = 20

model = ChickenCounting().to(DEVICE)
print('load model success')

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1 - lr_decay)

logging.info('Begin training single view model...')
train_model(model, train_loader, val_loader)
logging.info('Finished training single view model.')

In [ ]:
def evaluate(model, val_loader, device, scale):
    model.eval()

    mse = 0.0
    mae = 0.0
    predict_num = 0.0
    true_num = 0.0
    rate = 0.0

    with torch.no_grad():  # Disable gradient calculation for inference
        for i, data in enumerate(val_loader, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()  # Move inputs to device and convert to float
            targets = targets.to(device).float()  # Move targets to device and convert to float

            # Get the model predictions
            outputs = model(inputs) / scale  # Adjusting for the scaling factor

            # Convert tensors to numpy for visualization and metrics calculation
            inputs_np = inputs.cpu().numpy()  # Convert inputs to numpy
            targets_np = targets.cpu().numpy()  # Convert targets to numpy
            outputs_np = outputs.cpu().numpy()  # Convert outputs to numpy
            # imshow_res(inputs_np, targets_np, outputs_np, scale)  # Uncomment to visualize results

            # Calculate true and predicted sums for comparison
            t = np.sum((targets[0] / scale).cpu().numpy().squeeze())  # Ground truth sum
            g = np.sum(outputs.cpu().numpy().squeeze())  # Predicted sum
            print(f'NO.{i}   true_sum={t}, get_sum={g}, abs={abs(t - g)}, rate={abs(1 - g / t)}')

            # Update metrics
            predict_num += g
            true_num += t
            rate += abs(1 - g / t)
            mae += abs(t - g)
            mse += abs(t - g) * abs(t - g)

    # Calculate average metrics across all batches
    mae /= len(val_loader)
    mse /= len(val_loader)
    predict_num /= len(val_loader)
    true_num /= len(val_loader)
    rate /= len(val_loader)

    # Log the results
    logging.info(
        f'test ---- Score: {math.exp(-rate):.3f}, MSE: {mse:.4f}, MAE: {mae:.4f}, Chicken_avg: {predict_num:.4f}')
    return math.exp(-rate)

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth", map_location=DEVICE))
model.to(DEVICE)
evaluate(model, val_loader, DEVICE, 100)

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="validation")

def collate_fn(batch): # The test datasets will not provide target densities
    return torch.stack([image_transform(item["image"]) for item in batch])

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_a = np.concatenate(predictions, axis=0)

del test_dataset
del test_loader
del predictions

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="test")  
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_b = np.concatenate(predictions, axis=0)

np.savez('submission.npz', pred_a=pred_a, pred_b=pred_b)

In [ ]:
%run /kaggle/input/datasets/alengevorgyan/aaaaaaaaaaaaa/metrics.py